# CrisisOps GRPO Training Notebook

This notebook is the Phase 5 training pipeline for CrisisOps. Its job is to turn the environment into evidence for the hackathon's reward-improvement score: train Qwen3-8B with GRPO, log terminal rewards from the layered judge system, and save clean PNG plots for the README.

CrisisOps is not a static QA dataset. Each completion is interpreted as a buddy-pair incident response trajectory. The reward function runs that trajectory inside `CrisisOpsEnv`, then returns the terminal reward and judge breakdown to GRPO.

## 1. Install Training Stack

Run this on an A100 80GB runtime. We install Unsloth for QLoRA acceleration, vLLM for fast generation, TRL for `GRPOTrainer`, OpenEnv for compatibility, and plotting dependencies for the hackathon evidence images.

In [ ]:
%pip install -U pip
%pip install -U unsloth vllm trl transformers accelerate peft bitsandbytes datasets wandb openenv-core[core] matplotlib seaborn pandas


## 2. Imports and Repository Path

The notebook is designed to run from the repository root after cloning or uploading the project. If Colab starts elsewhere, set `REPO_ROOT` to the folder containing `crisisops_env`.

In [ ]:
import gc
import json
import os
import random
import re
import sys
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
import wandb
from datasets import Dataset

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "crisisops_env").exists():
    # Colab users can uncomment and edit this if they keep the repo in Drive.
    # REPO_ROOT = Path('/content/scalar_openenv_meta')
    raise FileNotFoundError("Run this notebook from the repository root containing crisisops_env/.")
sys.path.insert(0, str(REPO_ROOT))

from crisisops_env import Action, BuddyAction, BuddyFeedback, CrisisOpsEnv
from crisisops_env.models import Observation

torch.backends.cuda.matmul.allow_tf32 = True
sns.set_theme(style="whitegrid", context="talk")
print("Repo root:", REPO_ROOT)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 3. Load Qwen3-8B with Unsloth QLoRA

Qwen3-8B gives us enough reasoning capacity for noisy incident response while still fitting the A100 budget with 4-bit QLoRA. We keep sequence length at 4096 because prompts contain observability context and completions include `<think>` plus structured buddy actions.

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen3-8B"
MAX_SEQ_LENGTH = 4096
LORA_RANK = 32

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    fast_inference=True,
    max_lora_rank=LORA_RANK,
    gpu_memory_utilization=0.70,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=LORA_RANK,
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Loaded", MODEL_NAME)


## 4. Prompt Format

The model must output a complete buddy-pair trajectory. The reward bridge will parse `<actions>[...]</actions>` as a list of `BuddyAction` dictionaries and run them in `CrisisOpsEnv`. This keeps GRPO simple: one prompt, one completion, one environment rollout reward.

In [ ]:
ACTION_SCHEMA_HINT = {
    "primary_action": {
        "action_type": "query_metrics | read_logs | check_dependencies | run_healthcheck | restart_service | scale_service | rollback_config | drain_connections | set_rate_limit | diagnose",
        "target_service": "api_gateway | auth_service | user_db | order_service | payment_service | null",
        "parameters": {"root_cause": "...", "root_cause_service": "...", "severity": "P1|P2|P3"},
    },
    "buddy_feedback": {
        "feedback_type": "APPROVE | SUGGEST_ALTERNATIVE | FLAG_RISK",
        "rationale": "short reason",
        "risk_flags": ["optional risk labels"],
        "diagnosis": {"root_cause": "...", "root_cause_service": "...", "severity": "P1|P2|P3"},
    },
}

SYSTEM_INSTRUCTIONS = """You are CrisisOps, a two-agent SRE buddy pair.
Primary SRE proposes actions. Buddy SRE reviews each action before execution.
Investigate evidence before risky remediations. Avoid red herring services.
Restore the system before diagnosis. End with diagnose and independent buddy diagnosis.
Return only this format:
<think>brief private reasoning</think>
<actions>[JSON list of BuddyAction objects]</actions>
"""

def observation_to_prompt(obs: Observation, episode_id: int, difficulty: str) -> str:
    payload = {
        "episode_id": episode_id,
        "difficulty": difficulty,
        "system_overview": obs.system_overview,
        "recent_alerts": obs.recent_alerts,
        "action_result": obs.action_result,
        "time_remaining": obs.time_remaining,
        "available_actions": obs.available_actions,
        "available_feedback": obs.available_feedback,
        "action_schema_hint": ACTION_SCHEMA_HINT,
    }
    return SYSTEM_INSTRUCTIONS + "\nIncident observation:\n" + json.dumps(payload, indent=2)

def curriculum_difficulty(episode_idx: int) -> str:
    if episode_idx < 100:
        return "easy"
    if episode_idx < 220:
        return random.choice(["easy", "medium"])
    return random.choice(["medium", "hard"])


## 5. Dataset: Procedural Curriculum Prompts

The dataset stores initial observations, seeds, and difficulty. The reward function replays each completion against the same seeded incident so rewards are deterministic and comparable across GRPO groups.

In [ ]:
NUM_TRAIN_EPISODES = 360
BASE_SEED = 20260425

rows = []
for episode_idx in range(NUM_TRAIN_EPISODES):
    difficulty = curriculum_difficulty(episode_idx)
    seed = BASE_SEED + episode_idx
    env = CrisisOpsEnv()
    obs = env.reset(seed=seed, difficulty=difficulty)
    scenario = env.state().scenario
    rows.append({
        "prompt": observation_to_prompt(obs, episode_idx, difficulty),
        "seed": seed,
        "difficulty": difficulty,
        "scenario_type": scenario.scenario_type if scenario else "unknown",
    })

train_dataset = Dataset.from_list(rows)
print(train_dataset)
print(train_dataset[0]["prompt"][:1200])


## 6. Reward Bridge: Completion to CrisisOps Rollout

This is the core environment integration. The model writes actions; we parse them into `BuddyAction` objects, execute them in the environment, and return the terminal reward. We also record root-cause accuracy, process quality, and damage audit for the PNG plots.

In [ ]:
TRAINING_METRICS: List[Dict[str, Any]] = []

def completion_to_text(completion: Any) -> str:
    if isinstance(completion, str):
        return completion
    if isinstance(completion, list):
        if completion and isinstance(completion[-1], dict):
            return str(completion[-1].get("content", ""))
        return "\n".join(map(str, completion))
    if isinstance(completion, dict):
        return str(completion.get("content", completion))
    return str(completion)

def extract_json_actions(text: str) -> Tuple[List[BuddyAction], Optional[str]]:
    match = re.search(r"<actions>\s*(.*?)\s*</actions>", text, flags=re.DOTALL | re.IGNORECASE)
    raw = match.group(1) if match else text
    if not match:
        bracket = re.search(r"(\[.*\]|\{.*\})", text, flags=re.DOTALL)
        raw = bracket.group(1) if bracket else text
    try:
        payload = json.loads(raw)
    except Exception as exc:
        return [], f"json_parse_error: {exc}"
    if isinstance(payload, dict):
        payload = [payload]
    if not isinstance(payload, list):
        return [], "actions_payload_not_list"
    actions: List[BuddyAction] = []
    for item in payload:
        try:
            actions.append(BuddyAction.model_validate(item))
        except Exception as exc:
            return actions, f"buddy_action_validation_error: {exc}"
    return actions, None

def forced_wrong_diagnosis() -> BuddyAction:
    return BuddyAction(
        primary_action=Action(
            action_type="diagnose",
            parameters={"root_cause": "unknown", "root_cause_service": "api_gateway", "severity": "P3"},
        ),
        buddy_feedback=BuddyFeedback(feedback_type="APPROVE", rationale="Forced terminal fallback."),
    )

def run_completion_in_env(completion_text: str, seed: int, difficulty: str) -> Dict[str, Any]:
    env = CrisisOpsEnv()
    obs = env.reset(seed=int(seed), difficulty=difficulty)
    actions, parse_error = extract_json_actions(completion_text)
    if parse_error:
        return {
            "total_reward": 0.0,
            "root_cause_accuracy": 0.0,
            "process_quality": 0.0,
            "damage_audit": 0.0,
            "parse_error": parse_error,
            "scenario_type": env.state().scenario.scenario_type,
            "difficulty": difficulty,
        }
    for action in actions[: env.max_steps]:
        obs = env.step(action)
        if obs.done:
            break
    if not obs.done:
        obs = env.step(forced_wrong_diagnosis())
    rb = obs.reward_breakdown
    return {
        "total_reward": float(obs.reward or 0.0),
        "root_cause_accuracy": float(rb.root_cause_accuracy if rb else 0.0),
        "process_quality": float(rb.process_quality if rb else 0.0),
        "damage_audit": float(rb.damage_audit if rb else 0.0),
        "boss_score": float(rb.boss_score if rb else 0.0),
        "primary_reward": float(rb.primary_reward if rb else 0.0),
        "buddy_reward": float(rb.buddy_reward if rb else 0.0),
        "scenario_type": env.state().scenario.scenario_type,
        "difficulty": difficulty,
        "parse_error": None,
    }

def crisisops_reward_func(prompts, completions, seed, difficulty, scenario_type=None, trainer_state=None, **kwargs):
    rewards: List[float] = []
    for prompt, completion, seed_i, difficulty_i in zip(prompts, completions, seed, difficulty):
        metrics = run_completion_in_env(completion_to_text(completion), int(seed_i), str(difficulty_i))
        metrics["episode"] = len(TRAINING_METRICS)
        metrics["trainer_step"] = getattr(trainer_state, "global_step", None) if trainer_state is not None else None
        TRAINING_METRICS.append(metrics)
        rewards.append(metrics["total_reward"])
        if wandb.run is not None:
            wandb.log({
                "env/total_reward": metrics["total_reward"],
                "env/root_cause_accuracy": metrics["root_cause_accuracy"],
                "env/process_quality": metrics["process_quality"],
                "env/damage_audit": metrics["damage_audit"],
                "env/boss_score": metrics.get("boss_score", 0.0),
            }, commit=False)
    return rewards


## 7. Smoke Test the Reward Bridge

Before spending A100 time, verify that a known-good scripted trajectory gets a nonzero reward and that parser failures get zero. This is the fastest way to catch schema drift.

In [ ]:
def scripted_memory_leak_completion() -> str:
    actions = [
        {"primary_action": {"action_type": "query_metrics", "target_service": "api_gateway", "parameters": {}}, "buddy_feedback": {"feedback_type": "APPROVE", "rationale": "Start at customer symptom."}},
        {"primary_action": {"action_type": "query_metrics", "target_service": "auth_service", "parameters": {}}, "buddy_feedback": {"feedback_type": "APPROVE", "rationale": "Check likely upstream dependency."}},
        {"primary_action": {"action_type": "read_logs", "target_service": "auth_service", "parameters": {}}, "buddy_feedback": {"feedback_type": "APPROVE", "rationale": "Read root logs before remediation."}},
        {"primary_action": {"action_type": "restart_service", "target_service": "auth_service", "parameters": {}}, "buddy_feedback": {"feedback_type": "FLAG_RISK", "rationale": "State-changing but supported by memory evidence.", "risk_flags": ["state_change"]}},
        {"primary_action": {"action_type": "diagnose", "target_service": None, "parameters": {"root_cause": "memory_leak:auth_service", "root_cause_service": "auth_service", "severity": "P2"}}, "buddy_feedback": {"feedback_type": "APPROVE", "rationale": "Evidence supports diagnosis.", "diagnosis": {"root_cause": "memory_leak:auth_service", "root_cause_service": "auth_service", "severity": "P2"}}},
    ]
    return "<think>Investigate, remediate, diagnose.</think>\n<actions>" + json.dumps(actions) + "</actions>"

smoke = run_completion_in_env(scripted_memory_leak_completion(), seed=20260432, difficulty="easy")
print(smoke)
assert smoke["total_reward"] > 0.5, smoke


## 8. GRPO Configuration

On A100 80GB, group size `num_generations=4` is a good starting point for stable relative preference signals without blowing up rollout cost. Increase to 8 if VRAM and wall-clock time allow. W&B is enabled because the final README needs reward curves and judge-component plots.

In [ ]:
from trl import GRPOConfig, GRPOTrainer

WANDB_PROJECT = "crisisops-openenv-grpo"
wandb.init(project=WANDB_PROJECT, name="qwen3-8b-buddy-judges", config={
    "model": MODEL_NAME,
    "max_seq_length": MAX_SEQ_LENGTH,
    "lora_rank": LORA_RANK,
    "num_train_episodes": NUM_TRAIN_EPISODES,
    "curriculum": "easy first 100, then easy/medium, then medium/hard",
})

grpo_kwargs = dict(
    output_dir="checkpoints/crisisops-qwen3-8b-grpo",
    learning_rate=5e-6,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.01,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    bf16=True,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    num_generations=4,
    max_prompt_length=2048,
    max_completion_length=1024,
    max_steps=300,
    beta=0.02,
    temperature=0.8,
    logging_steps=5,
    save_steps=50,
    report_to="wandb",
    use_vllm=True,
    vllm_mode="colocate",
    vllm_gpu_memory_utilization=0.35,
    log_completions=True,
)

try:
    training_args = GRPOConfig(**grpo_kwargs)
except TypeError as exc:
    print("GRPOConfig rejected a newer argument, retrying with conservative args:", exc)
    for key in ["vllm_gpu_memory_utilization", "log_completions", "temperature"]:
        grpo_kwargs.pop(key, None)
    training_args = GRPOConfig(**grpo_kwargs)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    reward_funcs=crisisops_reward_func,
    train_dataset=train_dataset,
)


## 9. Train

This is the expensive cell. On the planned 12-hour A100 budget, start with 300 GRPO steps. If reward plateaus early, stop and move to Phase 6 plots and qualitative examples.

In [ ]:
assert torch.cuda.is_available(), "Phase 5 training should run on an A100 GPU runtime."
train_result = trainer.train()
trainer.save_model("checkpoints/crisisops-qwen3-8b-grpo/final")
tokenizer.save_pretrained("checkpoints/crisisops-qwen3-8b-grpo/final")
print(train_result)


## 10. Save Training Metrics

The reward function collected one row per generated completion. We save it as CSV so Phase 6 can re-plot or audit training evidence without rerunning GRPO.

In [ ]:
metrics_df = pd.DataFrame(TRAINING_METRICS)
if metrics_df.empty:
    raise RuntimeError("No training metrics collected. Run trainer.train() before plotting.")

metrics_df["episode"] = range(len(metrics_df))
metrics_df["reward_ma20"] = metrics_df["total_reward"].rolling(20, min_periods=1).mean()
metrics_df.to_csv("training_metrics.csv", index=False)
print(metrics_df.tail())
print("Saved training_metrics.csv with", len(metrics_df), "rollouts")


## 11. Hackathon Evidence Plots

These two PNGs directly target the 20% reward-improvement judging criterion. They use labeled axes, readable titles, high DPI, and the exact judge components we want to discuss in the README.

In [ ]:
plt.figure(figsize=(12, 7))
sns.lineplot(data=metrics_df, x="episode", y="total_reward", alpha=0.30, label="Episode reward")
sns.lineplot(data=metrics_df, x="episode", y="reward_ma20", linewidth=3, label="20-rollout moving average")
plt.xlabel("Generated Environment Rollout")
plt.ylabel("Terminal Total Reward")
plt.title("CrisisOps GRPO Training: Total Reward Over Rollouts")
plt.ylim(-0.02, 1.05)
plt.legend()
plt.tight_layout()
plt.savefig("reward_curve.png", dpi=300, bbox_inches="tight")
plt.show()

component_cols = ["root_cause_accuracy", "process_quality", "damage_audit"]
component_df = metrics_df[["episode"] + component_cols].copy()
for col in component_cols:
    component_df[col] = component_df[col].rolling(20, min_periods=1).mean()
long_components = component_df.melt(id_vars="episode", var_name="Judge Component", value_name="Score")

plt.figure(figsize=(12, 7))
sns.lineplot(data=long_components, x="episode", y="Score", hue="Judge Component", linewidth=3)
plt.xlabel("Generated Environment Rollout")
plt.ylabel("20-Rollout Moving Average Score")
plt.title("CrisisOps GRPO Training: Layered Judge Breakdown")
plt.ylim(-0.02, 1.05)
plt.legend(title="Judge Component")
plt.tight_layout()
plt.savefig("judge_breakdown.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved reward_curve.png and judge_breakdown.png")


## 12. Optional Baseline vs Trained Evaluation

After training, run a small held-out evaluation and save a third plot for Phase 6. This is not a substitute for the training curves; it is a clearer before/after story for the README.

In [ ]:
def summarize_success(df: pd.DataFrame, label: str) -> Dict[str, Any]:
    return {
        "policy": label,
        "mean_reward": float(df["total_reward"].mean()),
        "success_rate": float((df["total_reward"] >= 0.70).mean()),
        "root_cause_accuracy": float(df["root_cause_accuracy"].mean()),
    }

# Use the first 20 percent of rollouts as a rough early-training baseline and the last 20 percent as trained behavior.
window = max(10, len(metrics_df) // 5)
comparison = pd.DataFrame([
    summarize_success(metrics_df.head(window), "Early training"),
    summarize_success(metrics_df.tail(window), "Late training"),
])
comparison.to_csv("success_rate_comparison.csv", index=False)

plt.figure(figsize=(9, 6))
sns.barplot(data=comparison, x="policy", y="success_rate")
plt.xlabel("Policy Snapshot")
plt.ylabel("Success Rate (Reward >= 0.70)")
plt.title("CrisisOps: Early vs Late Training Success Rate")
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig("success_rate_comparison.png", dpi=300, bbox_inches="tight")
plt.show()
comparison


## 13. Cleanup and Export Notes

Artifacts expected after a successful run:

- `training_metrics.csv`
- `reward_curve.png`
- `judge_breakdown.png`
- `success_rate_comparison.png`
- `checkpoints/crisisops-qwen3-8b-grpo/final/`

For Phase 6, copy the PNG files into the README assets path and include captions explaining what improved: total reward, root-cause accuracy, process quality, and damage avoidance.

In [ ]:
if wandb.run is not None:
    wandb.log({
        "artifacts/reward_curve": wandb.Image("reward_curve.png"),
        "artifacts/judge_breakdown": wandb.Image("judge_breakdown.png"),
        "artifacts/success_rate_comparison": wandb.Image("success_rate_comparison.png"),
    })
    wandb.finish()

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
